# Tata Motors — Global LightGBM v6: Z-Scaled Cross-Learning + Advanced Accuracy Stack
**Architecture:** Per-model Z-score standardisation → Global LightGBM (regression) → MinT-style reconciliation  
**Target:** WAPE ≤ 15% per model after reconciliation  
**Data window:** Jan 2020 – latest  
**Changes from v5:**
1. **Per-model Z-scaling** of target before lag/feature build — strips volume differences, forces model to learn pure behaviour patterns  
2. **Objective: `regression` (L2)** — Tweedie dropped (incompatible with negative Z-scores)  
3. **Inverse transform at inference** — Z-scores → raw units before WAPE & PO output  
4. **Recency-weighted training samples** — exponential decay (λ=0.05) gives recent months higher loss weight  
5. **Extended EWM momentum features** — α=0.3/0.6, plus acceleration (lag1 − lag2)  
6. **Zero-demand imputation flag** — `is_structural_zero` distinguishes launch gaps from true zeros  
7. **MinT-inspired WLS reconciliation** — weight by inverse variance instead of flat proportional scaling  
8. **Model-cluster features** — KMeans(k=4) cluster label baked in as extra categorical  
9. **Optuna: 80 trials with pruning** — CMASSampler for faster convergence  
10. **Post-processing bias correction** — multiplicative scale factor learned on CV residuals per model

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, json, os
warnings.filterwarnings('ignore')

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

RANDOM_SEED      = 42
N_OPTUNA_TRIALS  = 20          # ↑ from 60
HORIZONS         = list(range(1, 13))
DATA_PATH        = 'cleaned_car_sales.csv'
TRAIN_START      = '2020-01-01'
OUTLIER_CAP_PCT  = 0.90
EWM_DECAY        = 0.05        # λ for recency sample weights
N_CLUSTERS       = 4           # KMeans model clusters

np.random.seed(RANDOM_SEED)
print('Imports OK  |  LightGBM:', lgb.__version__, ' Optuna:', optuna.__version__)


Imports OK  |  LightGBM: 4.6.0  Optuna: 4.8.0


## 1. Load, Filter & Cap Outlier
Same 2020-onward window and 90th-percentile proportional cap from v5.

In [4]:
df = pd.read_csv("/kaggle/input/datasets/sauravraj23/cleaned-car-sales/cleaned_car_sales.csv")
df['year_month'] = pd.to_datetime(df['year_month'])
df['sales_date'] = pd.to_datetime(df['sales_date'])
df = df[df['year_month'] >= TRAIN_START].copy()

# ── Cap OEM-level outlier proportionally across models ──
oem_raw   = df.groupby('year_month')['sales_unit'].sum()
cap_val   = oem_raw.quantile(OUTLIER_CAP_PCT)
bad_months = oem_raw[oem_raw > cap_val].index
print(f'90th-pctile cap: {cap_val:.0f}  |  months capped: {list(bad_months.strftime("%Y-%m"))}')
for ym in bad_months:
    mask  = df['year_month'] == ym
    scale = cap_val / df.loc[mask, 'sales_unit'].sum()
    df.loc[mask, 'sales_unit'] = (df.loc[mask, 'sales_unit'] * scale).round()

# ── OEM monthly ──
oem = (
    df.groupby('year_month')['sales_unit'].sum()
    .reset_index().rename(columns={'year_month': 'ds', 'sales_unit': 'y'})
    .sort_values('ds').reset_index(drop=True)
)
full_idx = pd.DataFrame({'ds': pd.date_range(oem.ds.min(), oem.ds.max(), freq='MS')})
oem = full_idx.merge(oem, on='ds', how='left').assign(y=lambda x: x['y'].ffill())

# ── Per-model monthly ──
model_monthly = (
    df.groupby(['year_month', 'model'])['sales_unit'].sum()
    .reset_index().rename(columns={'year_month': 'ds', 'sales_unit': 'y'})
    .sort_values(['model', 'ds']).reset_index(drop=True)
)
MODELS = sorted(df['model'].unique().tolist())
print(f'OEM: {len(oem)} months  {oem.ds.min().date()} → {oem.ds.max().date()}')
print(f'Models ({len(MODELS)}): {MODELS}')
oem.tail(4)


90th-pctile cap: 462  |  months capped: ['2025-01', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-10']
OEM: 70 months  2020-01-01 → 2025-10-01
Models (18): ['Altroz', 'Bolt', 'Curvv', 'Harrier', 'Hexa', 'Indica', 'Indigo', 'Nano', 'Nexon', 'Nexon EV', 'Punch', 'Safari', 'Safari Storme', 'Tiago', 'Tiago EV', 'Tigor', 'Tigor EV', 'Zest']


,ds,y
66,2025-07-01,490
67,2025-08-01,483
68,2025-09-01,410
69,2025-10-01,266


## 2. Per-Model Z-Score Standardisation  ← **The Core Fix**

This is the missing link in v5. The global model was trying to draw universal tree splits across series ranging from 50 → 16,000 units/month. The scale difference was catastrophic for loss weighting.

**After Z-scaling:**  
- Nexon at 16,000 in a strong month → `+1.5σ`  
- Tigor EV at 450 in an equally strong month → `+1.5σ`  
- LightGBM now sees *identical feature space* and can apply the same split to both.

**Key engineering decisions:**
- σ is computed on the **full training window** (not train-only) because we're standardising the series, not the model output
- If σ = 0 (constant series), we set σ = 1 to avoid division by zero
- Lags and rolling features are built **on the scaled series** so all temporal context is also scale-invariant
- `is_structural_zero` flag identifies pre-launch / production-gap zeros vs true demand zeros

In [5]:
# ── Step 1: Build per-model scaling params ──
# 1. Explicitly define the list of all car models first
all_models = sorted(model_monthly['model'].unique())

# Create a dictionary to hold the max value for each car
scaling_params = {}

for car_model in all_models:
    s = model_monthly[model_monthly['model'] == car_model]['y']
    
    # Get the maximum sales volume ever recorded for this car
    max_val = s.max()
    
    # Prevent division by zero if a car truly has no sales
    if max_val == 0:
        max_val = 1.0 
        
    scaling_params[car_model] = {'max': max_val}

print("Scaling params (Max Volume):")
for m, p in scaling_params.items():
    print(f"  {m:30s}  Max={p['max']:8.1f}")

Scaling params (Max Volume):
  Altroz                          Max=    48.0
  Bolt                            Max=    50.0
  Curvv                           Max=    48.0
  Harrier                         Max=    49.0
  Hexa                            Max=    50.0
  Indica                          Max=    55.0
  Indigo                          Max=    58.0
  Nano                            Max=    49.0
  Nexon                           Max=    43.0
  Nexon EV                        Max=    40.0
  Punch                           Max=    40.0
  Safari                          Max=    41.0
  Safari Storme                   Max=    47.0
  Tiago                           Max=    51.0
  Tiago EV                        Max=    47.0
  Tigor                           Max=    51.0
  Tigor EV                        Max=    53.0
  Zest                            Max=    58.0


## 3. Model Clustering (KMeans, k=4)
Group car models by their *scaled* behavioural signature (mean Z, std Z, trend slope, autocorrelation lag-12).  
The cluster label becomes an extra categorical feature — it lets LightGBM apply distinct splits for  
e.g. "high-volume stable" vs "low-volume volatile" models without fully separating them.

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

# Ensure we have the consistent list of models
all_models = sorted(model_monthly['model'].unique())

# Assuming N_CLUSTERS was defined earlier in your notebook, but we'll set a default just in case
N_CLUSTERS = 3 

def compute_model_profile(car_model):
    s = (
        model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
        .sort_values('ds').reset_index(drop=True)
    )
    
    # If not enough data, return a flat 0 profile
    if len(s) < 12:
        return [0.0, 0.0, 0.0, 0.0]
        
    # ── USE THE NEW 'max' SCALING ──
    max_val = scaling_params[car_model]['max']
    z = s['y'] / max_val
    
    # Calculate profile stats on the 0.0-1.0 relative scale
    trend_slope = np.polyfit(np.arange(len(z)), z, 1)[0]
    ac12 = z.autocorr(12) if len(z) > 13 else 0
    
    return [z.mean(), z.std(), trend_slope, float(ac12) if not np.isnan(ac12) else 0]

# Compute profiles and run K-Means
profiles = np.array([compute_model_profile(m) for m in all_models])
scaler_km = StandardScaler().fit(profiles)
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km.fit(scaler_km.transform(profiles))

# Save the cluster dictionary
MODEL_CLUSTER = {m: int(km.labels_[i]) for i, m in enumerate(all_models)}

print("Model clusters (Relative Scaling):")
for c in range(N_CLUSTERS):
    members = [m for m in all_models if MODEL_CLUSTER[m] == c]
    print(f"  Cluster {c}: {members}")

Model clusters (Relative Scaling):
  Cluster 0: ['Harrier', 'Indica', 'Indigo', 'Safari Storme', 'Tiago', 'Tiago EV']
  Cluster 1: ['Altroz', 'Hexa', 'Nano', 'Nexon', 'Nexon EV', 'Punch', 'Safari', 'Tigor', 'Tigor EV']
  Cluster 2: ['Bolt', 'Curvv', 'Zest']


## 4. Feature Maps (Festival, Macro, BS6/EV dummies)
Same Indian domain features from v4/v5, extended with EV segment flag and BS6 transition dummy.

In [7]:
FESTIVAL_MAP = {10: 2, 9: 1, 11: 1, 3: 1}  # Oct=Diwali, Sep/Nov=Navratri/post, Mar=year-end

EV_MODELS = {'Nexon EV', 'Tigor EV', 'Punch EV', 'Tiago EV'}  # adjust to actual model names

def get_ev_flag(car_model):
    for ev in EV_MODELS:
        if ev.lower() in car_model.lower():
            return 1
    return 0

# BS6 transition: April 2020 onward = BS6 era (1), before = 0
def bs6_flag(ds):
    return (ds >= pd.Timestamp('2020-04-01')).astype(int)

# FAME-II subsidy window: active for EV models
def fame2_flag(ds):
    return ((ds >= pd.Timestamp('2019-04-01')) & (ds <= pd.Timestamp('2024-03-31'))).astype(int)

print("Feature maps defined.")


Feature maps defined.


## 5. Global Panel Builder — Z-Scaled  ← **Surgical Rewrite**

Key changes vs v5:
- Target (`y`) is **Z-scored per model** before any feature is built
- All lags/rolling stats operate on `y_scaled` — so lag_1 is "last month's Z-score", not raw units
- Added EWM features (α=0.3 slow, α=0.6 fast), momentum acceleration, and the `is_structural_zero` flag
- `model_cluster` added as second categorical alongside `model_enc`

In [8]:
FESTIVAL_MAP = {10: 2, 9: 1, 11: 1, 3: 1}
EWM_DECAY = 0.05  # Ensure your decay constant is set

def build_global_panel_zscaled(model_monthly: pd.DataFrame, horizon: int):
    """
    Builds a stacked global panel where the TARGET and all LAG features
    are Relative-scaled (0.0 to 1.0) per model using the Max value.
    (Kept the function name 'zscaled' so it doesn't break your other loops!)
    """
    frames = []
    all_models = sorted(model_monthly['model'].unique())
    model_enc  = {m: i for i, m in enumerate(all_models)}

    for car_model in all_models:
        s = (
            model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
            .sort_values('ds').reset_index(drop=True)
        )
        full_r = pd.DataFrame({'ds': pd.date_range(s.ds.min(), s.ds.max(), freq='MS')})
        s = full_r.merge(s, on='ds', how='left')

        # ── Structural-zero imputation ──
        s['is_structural_zero'] = ((s['y'].isna()) | (s['y'] == 0)).astype(int)
        s['y'] = s['y'].fillna(0)

        if s['y'].sum() == 0 or len(s) < 18:
            continue

        # ── NEW RELATIVE SCALING (Replaces old Z-Scaling) ──
        max_val = scaling_params[car_model]['max']
        s['y_scaled'] = s['y'] / max_val

        # ── Calendar features ──
        s['month']   = s['ds'].dt.month
        s['quarter'] = s['ds'].dt.quarter
        s['year']    = s['ds'].dt.year
        s['trend']   = np.arange(len(s))

        # ── Model identity ──
        s['model_enc']     = model_enc[car_model]
        s['model_cluster'] = MODEL_CLUSTER[car_model]
        s['is_ev']         = get_ev_flag(car_model)

        # ── Domain dummies ──
        s['bs6_era']         = bs6_flag(s['ds']).values
        s['fame2_active']    = fame2_flag(s['ds']).values
        s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
        s['is_q4_india']     = s['month'].isin([1, 2, 3]).astype(int)
        s['is_festive_season'] = s['month'].isin([9, 10, 11]).astype(int)

        # ── Lag features on SCALED series ──
        for lag in [1, 2, 3, 4, 6, 9, 12, 24]:
            s[f'lag_{lag}'] = s['y_scaled'].shift(lag)

        # ── Rolling stats on SCALED series ──
        for w in [3, 6, 12]:
            rolled = s['y_scaled'].shift(1).rolling(w)
            s[f'roll_mean_{w}'] = rolled.mean()
            s[f'roll_std_{w}']  = rolled.std()
            s[f'roll_max_{w}']  = rolled.max()

        # ── EWM momentum (slow + fast) on SCALED series ──
        s['ewm_slow'] = s['y_scaled'].shift(1).ewm(alpha=0.3, adjust=False).mean()
        s['ewm_fast'] = s['y_scaled'].shift(1).ewm(alpha=0.6, adjust=False).mean()
        s['ewm_diff'] = s['ewm_fast'] - s['ewm_slow']   # momentum crossover signal

        # ── Acceleration: (lag1 - lag2) ──
        s['acceleration'] = s['lag_1'] - s['lag_2']

        # ── YoY on scaled series ──
        s['yoy_scaled'] = s['y_scaled'].shift(12).pct_change(1)
        s['lag12_scaled'] = s['y_scaled'].shift(12)

        # ── Target = Scaled value h months ahead ──
        s['horizon'] = horizon
        s['target']  = s['y_scaled'].shift(-horizon)

        # ── Recency weights: exponential decay, newest row = 1.0 ──
        n = len(s)
        s['sample_weight'] = np.exp(EWM_DECAY * np.arange(n))
        s['sample_weight'] /= s['sample_weight'].max()

        required = [f'lag_{l}' for l in [1, 2, 3, 6]]
        s = s.dropna(subset=['target'] + required).reset_index(drop=True)
        frames.append(s)

    if not frames:
        return None, None, None, None, None, None

    panel = pd.concat(frames, ignore_index=True)

    feature_cols = [c for c in panel.columns if c not in
                    ['ds', 'y', 'y_scaled', 'target', 'sample_weight']]
    cat_features = ['model_enc', 'model_cluster']
    
    # Simplified ds tracking to prevent KeyError string replace bugs
    ds_tracking = panel[['ds', 'model_enc']].copy()

    return (
        panel[feature_cols].values,
        panel['target'].values,
        ds_tracking,
        feature_cols,
        cat_features,
        panel['sample_weight'].values,
    )

# Quick shape check
_X, _y, _, _fn, _cf, _sw = build_global_panel_zscaled(model_monthly, horizon=1)
print(f'Global Scaled panel (h=1): {_X.shape[0]} rows × {_X.shape[1]} features')
print(f'Features: {_fn}')
print(f'Categorical: {_cf}')
print(f'Target range (Relative Scale): [{_y.min():.2f}, {_y.max():.2f}]')
print(f'Sample weights range: [{_sw.min():.3f}, {_sw.max():.3f}]')

Global Scaled panel (h=1): 1134 rows × 37 features
Features: ['is_structural_zero', 'month', 'quarter', 'year', 'trend', 'model_enc', 'model_cluster', 'is_ev', 'bs6_era', 'fame2_active', 'festival_intensity', 'is_q4_india', 'is_festive_season', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_6', 'lag_9', 'lag_12', 'lag_24', 'roll_mean_3', 'roll_std_3', 'roll_max_3', 'roll_mean_6', 'roll_std_6', 'roll_max_6', 'roll_mean_12', 'roll_std_12', 'roll_max_12', 'ewm_slow', 'ewm_fast', 'ewm_diff', 'acceleration', 'yoy_scaled', 'lag12_scaled', 'horizon']
Categorical: ['model_enc', 'model_cluster']
Target range (Relative Scale): [0.02, 1.00]
Sample weights range: [0.043, 0.951]


In [9]:
# ── Correct panel builder (clean version with model_name and MAX scaling) ──
def build_global_panel_zscaled(model_monthly: pd.DataFrame, horizon: int):
    """
    Stacked global panel — target and lag features are Relative-scaled (0 to 1.0) per model.
    LightGBM sees ONLY relative signals, never raw units.
    """
    frames = []
    all_models = sorted(model_monthly['model'].unique())
    model_enc  = {m: i for i, m in enumerate(all_models)}

    for car_model in all_models:
        s = (
            model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
            .sort_values('ds').reset_index(drop=True)
        )
        full_r = pd.DataFrame({'ds': pd.date_range(s.ds.min(), s.ds.max(), freq='MS')})
        s = full_r.merge(s, on='ds', how='left')
        s['is_structural_zero'] = ((s['y'].isna()) | (s['y'] == 0)).astype(int)
        s['y'] = s['y'].fillna(0)

        if s['y'].sum() == 0 or len(s) < 18:
            continue

        # ── NEW MAX SCALING (Replaces old Z-Scaling) ──
        max_val = scaling_params[car_model]['max']
        s['y_scaled'] = s['y'] / max_val

        s['month']    = s['ds'].dt.month
        s['quarter']  = s['ds'].dt.quarter
        s['year']     = s['ds'].dt.year
        s['trend']    = np.arange(len(s))
        s['model_enc']     = model_enc[car_model]
        s['model_cluster'] = MODEL_CLUSTER[car_model]
        s['model_name']    = car_model
        s['is_ev']         = get_ev_flag(car_model)
        s['bs6_era']       = bs6_flag(s['ds']).values
        s['fame2_active']  = fame2_flag(s['ds']).values
        s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
        s['is_q4_india']        = s['month'].isin([1, 2, 3]).astype(int)
        s['is_festive_season']  = s['month'].isin([9, 10, 11]).astype(int)

        for lag in [1, 2, 3, 4, 6, 9, 12, 24]:
            s[f'lag_{lag}'] = s['y_scaled'].shift(lag)
        for w in [3, 6, 12]:
            rolled = s['y_scaled'].shift(1).rolling(w)
            s[f'roll_mean_{w}'] = rolled.mean()
            s[f'roll_std_{w}']  = rolled.std()
            s[f'roll_max_{w}']  = rolled.max()

        s['ewm_slow']    = s['y_scaled'].shift(1).ewm(alpha=0.3, adjust=False).mean()
        s['ewm_fast']    = s['y_scaled'].shift(1).ewm(alpha=0.6, adjust=False).mean()
        s['ewm_diff']    = s['ewm_fast'] - s['ewm_slow']
        s['acceleration']  = s['lag_1'] - s['lag_2']
        s['yoy_scaled']    = s['y_scaled'].shift(12).pct_change(1)
        s['lag12_scaled']  = s['y_scaled'].shift(12)
        s['horizon']       = horizon
        s['target']        = s['y_scaled'].shift(-horizon)

        n = len(s)
        s['sample_weight'] = np.exp(EWM_DECAY * np.arange(n))
        s['sample_weight'] /= s['sample_weight'].max()

        required = [f'lag_{l}' for l in [1, 2, 3, 6]]
        s = s.dropna(subset=['target'] + required).reset_index(drop=True)
        frames.append(s)

    if not frames:
        return None, None, None, None, None, None

    panel = pd.concat(frames, ignore_index=True)
    feature_cols = [c for c in panel.columns if c not in
                    ['ds', 'y', 'y_scaled', 'target', 'sample_weight', 'model_name']]
    cat_features = ['model_enc', 'model_cluster']

    return (
        panel[feature_cols].values,
        panel['target'].values,
        panel[['ds', 'model_name']].copy(),
        feature_cols,
        cat_features,
        panel['sample_weight'].values,
    )

X, y, ds_info, feat_names, cat_feats, sw = build_global_panel_zscaled(model_monthly, horizon=1)
print(f'Panel (h=1): {X.shape[0]} rows × {X.shape[1]} features')
print(f'Categoricals: {cat_feats}')
print(f'Target Range (0 to 1): [{y.min():.2f}, {y.max():.2f}]')

Panel (h=1): 1134 rows × 37 features
Categoricals: ['model_enc', 'model_cluster']
Target Range (0 to 1): [0.02, 1.00]


## 6. Metrics

In [10]:
def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    return 0.0 if denom < 1e-8 else np.sum(np.abs(y_true - y_pred)) / denom * 100

def bias(y_true, y_pred):
    return np.mean(y_pred - y_true)

def mase(y_true, y_pred, y_train, m=1):
    naive_mae = np.mean(np.abs(np.diff(y_train, n=m))) + 1e-8
    return np.mean(np.abs(y_true - y_pred)) / naive_mae


## 7. Walk-Forward CV Splits (unchanged from v5)

In [11]:
def walk_forward_splits(n_months, n_folds=3, test_size=6, min_train=24):
    splits = []
    for i in range(n_folds, 0, -1):
        test_end   = n_months - (i - 1) * test_size
        test_start = test_end - test_size
        if test_start < min_train:
            continue
        splits.append((test_start, test_end))
    return splits

oem_months = len(oem)
for i, (ts, te) in enumerate(walk_forward_splits(oem_months)):
    print(f'Fold {i+1}: train [0→{ts-1}]  test [{ts}→{te-1}]  '
          f'({oem.ds.iloc[ts].strftime("%Y-%m")} → {oem.ds.iloc[min(te-1, oem_months-1)].strftime("%Y-%m")})')


Fold 1: train [0→51]  test [52→57]  (2024-05 → 2024-10)
Fold 2: train [0→57]  test [58→63]  (2024-11 → 2025-04)
Fold 3: train [0→63]  test [64→69]  (2025-05 → 2025-10)


## 8. Global LightGBM Tuner — Regression Objective + Sample Weights

Key changes vs v5:
- `objective: 'regression'` (not Tweedie) — target is Z-scored, can be negative
- `sample_weight` passed to `fit()` — exponential recency decay rewards recent months
- 80 Optuna trials with `TPESampler`
- Validation metric: **WAPE computed on inverse-transformed predictions** (raw units) for true accuracy signal

In [12]:
def tune_global_lgbm_zscaled(X_tr, y_tr, X_val, y_val, cat_idx,
                                sw_tr, model_names_val, horizon,
                                n_trials=N_OPTUNA_TRIALS):
    """
    Tunes a regression LightGBM on Z-scored targets.
    Validation WAPE is computed in RAW UNIT space after inverse transform.
    """
    # Pre-compute inverse transform lookup for val rows
    # model_names_val is a Series/array aligned with X_val rows
    def inverse_transform_vec(z_arr, model_name_arr):
        raw = np.zeros(len(z_arr))
        for i, (z, m) in enumerate(zip(z_arr, model_name_arr)):
            sp = scaling_params[m]
            raw[i] = z * sp['std'] + sp['mean']
        return np.clip(raw, 0, None)

    def objective(trial):
        params = {
            'objective':         'regression',
            'n_estimators':      trial.suggest_int('n_estimators', 300, 1200),
            'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.12, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 15.0, log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
            'max_depth':         trial.suggest_int('max_depth', 4, 10),
            'verbose': -1, 'n_jobs': -1, 'random_state': RANDOM_SEED,
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr,
              sample_weight=sw_tr,
              categorical_feature=cat_idx)
        z_pred = m.predict(X_val)
        # Inverse transform to raw units for metric
        raw_pred   = inverse_transform_vec(z_pred,   model_names_val)
        raw_actual = inverse_transform_vec(y_val,     model_names_val)
        return wape(raw_actual, raw_pred)

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = {
        **study.best_params,
        'objective': 'regression',
        'verbose': -1, 'n_jobs': -1, 'random_state': RANDOM_SEED,
    }
    final = lgb.LGBMRegressor(**best_params)
    final.fit(X_tr, y_tr, sample_weight=sw_tr, categorical_feature=cat_idx)
    return final, study.best_value

print('Tuner defined.')


Tuner defined.


## 9. Train All 12 Global Models (Z-scaled regression)

In [13]:
def inverse_transform_vec(z_arr, model_name_arr):
    """Un-scales the relative targets back to raw car units for metric calculation"""
    raw = np.zeros_like(z_arr)
    for i, (z, m) in enumerate(zip(z_arr, model_name_arr)):
        sp = scaling_params[m]
        # ── NEW MAX INVERSE TRANSFORM ──
        raw[i] = z * sp['max'] 
    return np.clip(raw, 0, None)

def tune_global_lgbm_zscaled(X_tr, y_tr, X_val, y_val, cat_idx, sw_tr, model_names_val, horizon, n_trials=30):
    def objective(trial):
        params = {
            'objective': 'tweedie',                 # Critical for 0.0-1.0 scaling
            'tweedie_variance_power': 1.5,          # Compound Poisson-Gamma
            'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 16, 96),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
            'verbose': -1, 'n_jobs': 4, 'random_state': RANDOM_SEED
        }
        
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, categorical_feature=cat_idx, sample_weight=sw_tr)
        
        z_pred = m.predict(X_val)
        
        # Inverse transform to raw units to calculate the true real-world error
        raw_pred   = inverse_transform_vec(z_pred, model_names_val)
        raw_actual = inverse_transform_vec(y_val, model_names_val)
        
        return wape(raw_actual, raw_pred)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    # Train Final Model on this fold with the best found parameters
    best_params = {
        **study.best_params,
        'objective': 'tweedie',
        'tweedie_variance_power': 1.5,
        'verbose': -1, 'n_jobs': 4, 'random_state': RANDOM_SEED
    }
    
    final = lgb.LGBMRegressor(**best_params)
    final.fit(X_tr, y_tr, categorical_feature=cat_idx, sample_weight=sw_tr)
    
    return final, study.best_value


# ── 2. The Training Loop (Calling the correct function) ──
def train_global_model_v6(model_monthly, horizon, n_folds=3):
    X, y, ds_info, feat_names, cat_feats, sw = build_global_panel_zscaled(model_monthly, horizon)
    if X is None:
        return None

    oem_dates    = sorted(ds_info['ds'].unique())
    date_to_idx  = {d: i for i, d in enumerate(oem_dates)}
    row_month_idx = ds_info['ds'].map(date_to_idx).values

    cat_idx = [feat_names.index(c) for c in cat_feats]
    splits  = walk_forward_splits(len(oem_dates), n_folds=n_folds)

    fold_metrics = []
    for fold_i, (ts, te) in enumerate(splits):
        tr_mask = row_month_idx < ts
        te_mask = (row_month_idx >= ts) & (row_month_idx < te)
        if tr_mask.sum() < 50 or te_mask.sum() < 5:
            continue
            
        # model names for val rows (needed for inverse transform in tuner)
        val_model_names = ds_info['model_name'].values[te_mask]

        m, best_wape = tune_global_lgbm_zscaled(
            X[tr_mask], y[tr_mask],
            X[te_mask], y[te_mask],
            cat_idx,
            sw[tr_mask],
            val_model_names,
            horizon,
        )
        fold_metrics.append({'fold': fold_i + 1, 'wape': round(best_wape, 2)})

    # Final model on all data minus last test window
    final_ts = splits[-1][0] if splits else len(oem_dates) - 6
    tr_mask  = row_month_idx < final_ts
    te_mask  = row_month_idx >= final_ts
    val_model_names = ds_info['model_name'].values[te_mask]
    
    final_model, _ = tune_global_lgbm_zscaled(
        X[tr_mask], y[tr_mask],
        X[te_mask], y[te_mask],
        cat_idx,
        sw[tr_mask],
        val_model_names,
        horizon,
    )

    return {
        'horizon':      horizon,
        'model':        final_model,
        'feat_names':   feat_names,
        'cat_idx':      cat_idx,
        'cat_feats':    cat_feats,
        'fold_metrics': fold_metrics,
    }


# ── 3. Execute Loop ──
# Explicitly define the horizons (1 to 12 months ahead)
HORIZONS = list(range(1, 13))

global_models = {}
for h in HORIZONS:
    print(f'\n── h={h} ──')
    result = train_global_model_v6(model_monthly, horizon=h)
    if result is None:
        print('  skipped')
        continue
    global_models[h] = result
    wapes = [fm['wape'] for fm in result['fold_metrics']]
    print(f'  fold WAPEs (raw units): {wapes}   mean={np.mean(wapes):.1f}%')

print('\nAll 12 global models trained.')


── h=1 ──
  fold WAPEs (raw units): [41.02, 39.0, 34.88]   mean=38.3%

── h=2 ──
  fold WAPEs (raw units): [41.16, 37.67, 34.79]   mean=37.9%

── h=3 ──
  fold WAPEs (raw units): [40.66, 38.61, 36.36]   mean=38.5%

── h=4 ──
  fold WAPEs (raw units): [40.64, 38.43, 35.15]   mean=38.1%

── h=5 ──
  fold WAPEs (raw units): [40.72, 37.64, 33.36]   mean=37.2%

── h=6 ──
  fold WAPEs (raw units): [40.39, 39.14, 36.68]   mean=38.7%

── h=7 ──
  fold WAPEs (raw units): [40.17, 39.42, 36.67]   mean=38.8%

── h=8 ──
  fold WAPEs (raw units): [39.18, 39.28, 37.01]   mean=38.5%

── h=9 ──
  fold WAPEs (raw units): [39.32, 38.94, 36.55]   mean=38.3%

── h=10 ──
  fold WAPEs (raw units): [40.63, 39.07, 37.41]   mean=39.0%

── h=11 ──
  fold WAPEs (raw units): [39.96, 39.28, 37.25]   mean=38.8%

── h=12 ──
  fold WAPEs (raw units): [39.92, 39.83, 37.14]   mean=39.0%

All 12 global models trained.


## 10. OEM Top-Down Model (unchanged — good anchor)
OEM series is large-volume and stable; no need to Z-scale it. Keep regression objective.

In [14]:
def build_oem_features(series, horizon):
    s = series.copy().reset_index(drop=True)
    s['month']   = s['ds'].dt.month
    s['quarter'] = s['ds'].dt.quarter
    s['year']    = s['ds'].dt.year
    s['trend']   = np.arange(len(s))
    for lag in [1, 2, 3, 4, 6, 9, 12]:
        s[f'lag_{lag}'] = s['y'].shift(lag)
    for w in [3, 6, 12]:
        r = s['y'].shift(1).rolling(w)
        s[f'roll_mean_{w}'] = r.mean()
        s[f'roll_std_{w}']  = r.std()
        s[f'roll_max_{w}']  = r.max()
    s['yoy_change']         = s['y'].shift(12).pct_change(1)
    s['lag12_val']          = s['y'].shift(12)
    s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
    s['is_q4_india']        = s['month'].isin([1, 2, 3]).astype(int)
    s['is_festive_season']  = s['month'].isin([9, 10, 11]).astype(int)
    s['horizon']            = horizon
    s['target']             = s['y'].shift(-horizon)
    req = [f'lag_{l}' for l in [1, 2, 3, 6]]
    s = s.dropna(subset=['target'] + req).reset_index(drop=True)
    feat_cols = [c for c in s.columns if c not in ['ds', 'y', 'target']]
    return s[feat_cols].values, s['target'].values, s['ds'].values, feat_cols


def tune_oem_lgbm(X_tr, y_tr, X_val, y_val, n_trials=N_OPTUNA_TRIALS):
    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 16, 96),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
            'verbose': -1, 'n_jobs': -1, 'random_state': RANDOM_SEED,
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr)
        return wape(y_val, np.clip(m.predict(X_val), 0, None))

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = lgb.LGBMRegressor(**{**study.best_params,
                                'verbose': -1, 'n_jobs': -1, 'random_state': RANDOM_SEED})
    best.fit(X_tr, y_tr)
    return best


print('Training OEM top-down models...')
oem_models = {}
for h in HORIZONS:
    X, y_oem, ds_oem, feat_oem = build_oem_features(oem, h)
    n = len(X)
    cut = n - 6
    oem_models[h] = tune_oem_lgbm(X[:cut], y_oem[:cut], X[cut:], y_oem[cut:])
    p = np.clip(oem_models[h].predict(X[cut:]), 0, None)
    print(f'  h={h:2d}: OEM WAPE = {wape(y_oem[cut:], p):.1f}%')
print('OEM models trained.')


Training OEM top-down models...
  h= 1: OEM WAPE = 19.0%
  h= 2: OEM WAPE = 20.2%
  h= 3: OEM WAPE = 11.6%
  h= 4: OEM WAPE = 21.6%
  h= 5: OEM WAPE = 27.6%
  h= 6: OEM WAPE = 19.2%
  h= 7: OEM WAPE = 19.8%
  h= 8: OEM WAPE = 19.1%
  h= 9: OEM WAPE = 12.4%
  h=10: OEM WAPE = 17.7%
  h=11: OEM WAPE = 17.4%
  h=12: OEM WAPE = 12.2%
OEM models trained.


## 11. Per-Model Multiplicative Bias Correction  ← **New in v6**

Even after Z-scaling, individual models may have a systematic bias (e.g., the global model consistently over-predicts Punch EV during its ramp-up phase).  
We learn a per-model **multiplicative scale factor** on the CV residuals and apply it at inference.

Formula: `bias_factor[model] = mean(actual_units) / mean(predicted_units)` over the CV test windows.  
This is clipped to [0.5, 2.0] to prevent outlier corrections from exploding.

In [16]:
def compute_bias_corrections(model_monthly, global_models, n_test=6):
    """
    Learn per-model multiplicative bias correction from CV test-window residuals.
    Returns dict: {car_model: scale_factor}
    """
    all_models = sorted(model_monthly['model'].unique())
    model_enc  = {m: i for i, m in enumerate(all_models)}
    bias_corr  = {m: 1.0 for m in all_models}

    # Use h=1 model as representative (can extend to per-horizon)
    h = 1
    if h not in global_models:
        return bias_corr

    gm   = global_models[h]
    feat = gm['feat_names']
    
    # Safe categorical index retrieval
    cidx = gm.get('cat_idx', 'auto')

    for car_model in all_models:
        sp = scaling_params[car_model]
        s = (
            model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
            .sort_values('ds').reset_index(drop=True)
        )
        full_r = pd.DataFrame({'ds': pd.date_range(s.ds.min(), s.ds.max(), freq='MS')})
        s = full_r.merge(s, on='ds', how='left').assign(y=lambda x: x['y'].fillna(0))
        
        if len(s) < 18 + n_test:
            continue

        s['is_structural_zero'] = (s['y'] == 0).astype(int)
        
        # ── NEW MAX SCALING ──
        s['y_scaled'] = s['y'] / sp['max']
        
        s['month']    = s['ds'].dt.month
        s['quarter']  = s['ds'].dt.quarter
        s['year']     = s['ds'].dt.year
        s['trend']    = np.arange(len(s))
        s['model_enc']     = model_enc[car_model]
        s['model_cluster'] = MODEL_CLUSTER[car_model]
        s['is_ev']         = get_ev_flag(car_model)
        s['bs6_era']       = bs6_flag(s['ds']).values
        s['fame2_active']  = fame2_flag(s['ds']).values
        s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
        s['is_q4_india']        = s['month'].isin([1, 2, 3]).astype(int)
        s['is_festive_season']  = s['month'].isin([9, 10, 11]).astype(int)
        
        for lag in [1, 2, 3, 4, 6, 9, 12, 24]:
            s[f'lag_{lag}'] = s['y_scaled'].shift(lag)
        for w in [3, 6, 12]:
            rolled = s['y_scaled'].shift(1).rolling(w)
            s[f'roll_mean_{w}'] = rolled.mean()
            s[f'roll_std_{w}']  = rolled.std()
            s[f'roll_max_{w}']  = rolled.max()
            
        s['ewm_slow']    = s['y_scaled'].shift(1).ewm(alpha=0.3, adjust=False).mean()
        s['ewm_fast']    = s['y_scaled'].shift(1).ewm(alpha=0.6, adjust=False).mean()
        s['ewm_diff']    = s['ewm_fast'] - s['ewm_slow']
        s['acceleration']  = s['lag_1'] - s['lag_2']
        s['yoy_scaled']    = s['y_scaled'].shift(12).pct_change(1)
        s['lag12_scaled']  = s['y_scaled'].shift(12)
        s['horizon']       = h
        s['target']        = s['y_scaled'].shift(-h)
        
        req = [f'lag_{l}' for l in [1, 2, 3, 6]]
        s = s.dropna(subset=['target'] + req).reset_index(drop=True)
        if len(s) < n_test + 1:
            continue

        te = s.iloc[-n_test:]
        missing_feats = [f for f in feat if f not in te.columns]
        for mf in missing_feats:
            te = te.copy(); te[mf] = 0

        z_pred = gm['model'].predict(te[feat].values, categorical_feature=cidx)
        
        # ── NEW INVERSE TRANSFORM ──
        raw_pred   = np.clip(z_pred * sp['max'], 0, None)
        raw_actual = np.clip(te['target'].values * sp['max'], 0, None)

        mean_actual = raw_actual.mean()
        mean_pred   = raw_pred.mean()
        if mean_pred > 1e-6:
            factor = mean_actual / mean_pred
            bias_corr[car_model] = float(np.clip(factor, 0.5, 2.0))

    return bias_corr

# Execute
bias_corrections = compute_bias_corrections(model_monthly, global_models)
print("Per-model bias correction factors:")
for m, f in sorted(bias_corrections.items()):
    status = '✓' if 0.85 <= f <= 1.15 else '⚠ LARGE'
    print(f"  {m:30s}  ×{f:.3f}  {status}")

Per-model bias correction factors:
  Altroz                          ×1.237  ⚠ LARGE
  Bolt                            ×0.770  ⚠ LARGE
  Curvv                           ×0.810  ⚠ LARGE
  Harrier                         ×0.850  ⚠ LARGE
  Hexa                            ×0.805  ⚠ LARGE
  Indica                          ×0.984  ✓
  Indigo                          ×1.163  ⚠ LARGE
  Nano                            ×1.082  ✓
  Nexon                           ×0.876  ✓
  Nexon EV                        ×1.091  ✓
  Punch                           ×0.890  ✓
  Safari                          ×1.170  ⚠ LARGE
  Safari Storme                   ×0.890  ✓
  Tiago                           ×1.005  ✓
  Tiago EV                        ×0.978  ✓
  Tigor                           ×1.013  ✓
  Tigor EV                        ×0.844  ⚠ LARGE
  Zest                            ×1.204  ⚠ LARGE


## 12. WLS (Inverse-Variance) Hierarchical Reconciliation  ← **New in v6**

V5 used flat proportional reconciliation: `f_i_reconciled = f_i × (OEM / Σf_i)`.  
This treated Nexon (high-confidence, high-volume) and a niche model (low-confidence, low-volume) identically.

**V6 uses Weighted Least Squares (MinT-inspired):**  
- Each model's weight = `1 / σ_i²` where σ_i is the model's historical residual variance  
- High-error models get down-weighted; high-confidence models (e.g., Nexon) dominate the reconciliation  
- The OEM anchor sum constraint is still enforced exactly

In [18]:
def compute_model_variance(model_monthly, global_models, n_test=6):
    """Compute per-model forecast error variance (σ²) from CV residuals."""
    
    all_models = sorted(model_monthly['model'].unique())
    model_enc  = {m: i for i, m in enumerate(all_models)}
    variances  = {m: 1.0 for m in all_models}

    h = 1
    if h not in global_models:
        return variances
        
    gm = global_models[h]
    feat = gm['feat_names']
    
    # Safely retrieve categorical index, default to 'auto' if missing
    cidx = gm.get('cat_idx', 'auto') 

    for car_model in all_models:
        sp = scaling_params[car_model]
        s = (
            model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
            .sort_values('ds').reset_index(drop=True)
        )
        
        full_r = pd.DataFrame({'ds': pd.date_range(s.ds.min(), s.ds.max(), freq='MS')})
        s = full_r.merge(s, on='ds', how='left').assign(y=lambda x: x['y'].fillna(0))
        
        if len(s) < 18 + n_test:
            continue

        s['is_structural_zero'] = (s['y'] == 0).astype(int)
        
        # ── RELATIVE SCALING ──
        s['y_scaled'] = s['y'] / sp['max']
        
        s['month']              = s['ds'].dt.month
        s['quarter']            = s['ds'].dt.quarter
        s['year']               = s['ds'].dt.year
        s['trend']              = np.arange(len(s))
        s['model_enc']          = model_enc[car_model]
        s['model_cluster']      = MODEL_CLUSTER[car_model]
        s['is_ev']              = get_ev_flag(car_model)
        s['bs6_era']            = bs6_flag(s['ds']).values
        s['fame2_active']       = fame2_flag(s['ds']).values
        s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
        s['is_q4_india']        = s['month'].isin([1, 2, 3]).astype(int)
        s['is_festive_season']  = s['month'].isin([9, 10, 11]).astype(int)
        
        for lag in [1, 2, 3, 4, 6, 9, 12, 24]:
            s[f'lag_{lag}'] = s['y_scaled'].shift(lag)
            
        for w in [3, 6, 12]:
            rolled = s['y_scaled'].shift(1).rolling(w)
            s[f'roll_mean_{w}'] = rolled.mean()
            s[f'roll_std_{w}']  = rolled.std()
            s[f'roll_max_{w}']  = rolled.max()
            
        s['ewm_slow']     = s['y_scaled'].shift(1).ewm(alpha=0.3, adjust=False).mean()
        s['ewm_fast']     = s['y_scaled'].shift(1).ewm(alpha=0.6, adjust=False).mean()
        s['ewm_diff']     = s['ewm_fast'] - s['ewm_slow']
        s['acceleration'] = s['lag_1'] - s['lag_2']
        s['yoy_scaled']   = s['y_scaled'].shift(12).pct_change(1)
        s['lag12_scaled'] = s['y_scaled'].shift(12)
        s['horizon']      = h
        s['target']       = s['y_scaled'].shift(-h)
        
        req = [f'lag_{l}' for l in [1, 2, 3, 6]]
        s = s.dropna(subset=['target'] + req).reset_index(drop=True)
        
        if len(s) < n_test + 1:
            continue

        te = s.iloc[-n_test:].copy()
        missing_feats = [f for f in feat if f not in te.columns]
        for mf in missing_feats:
            te[mf] = 0

        z_pred = gm['model'].predict(te[feat].values, categorical_feature=cidx)
        
        # ── MAX INVERSE TRANSFORM ──
        raw_pred   = np.clip(z_pred * sp['max'], 0, None)
        raw_actual = np.clip(te['target'].values * sp['max'], 0, None)
        
        residuals  = raw_actual - raw_pred
        
        # Add tiny constant (1e-6) to prevent DivisionByZero later in WLS weighting
        variances[car_model] = float(np.var(residuals) + 1e-6)

    return variances


# Print logic execution
model_variances = compute_model_variance(model_monthly, global_models)
print("Per-model forecast variances (lower = more reliable):")
for m, v in sorted(model_variances.items(), key=lambda x: x[1]):
    print(f"  {m:30s}  σ²={v:10.1f}")


def reconcile_wls(bu_raw_dict: dict, oem_total: float, variances: dict, bias_corr: dict) -> dict:
    """
    Optimal MinT (Diagonal) top-down reconciliation:
    1. Apply bias correction to raw BU forecasts
    2. Calculate the 'Gap' between OEM Top-Down and the BU sum
    3. Distribute the Gap proportionally to the variance (Uncertainty)
    """
    # Step 1: Apply bias correction
    bc_preds = {m: max(0.0, v * bias_corr.get(m, 1.0)) for m, v in bu_raw_dict.items()}
    
    # Step 2: Calculate the Total Gap
    sum_bu = sum(bc_preds.values())
    gap = oem_total - sum_bu
    
    # Step 3: Compute total variance 
    # (Use raw variance, NOT inverse variance. Higher variance = absorbs more of the gap)
    total_var = sum([variances.get(m, 1.0) for m in bc_preds.keys()])
    
    # Fallback to standard proportional scaling if variance is somehow zero
    if total_var < 1e-6:
        if sum_bu < 1e-6: return bc_preds
        return {m: max(0.0, round((v / sum_bu) * oem_total, 1)) for m, v in bc_preds.items()}

    # Step 4: Distribute the Gap based on Variance Weight
    reconciled = {}
    for m, v in bc_preds.items():
        # Weight = This model's variance / Total variance
        weight = variances.get(m, 1.0) / total_var
        
        # New Prediction = Base Prediction + (Gap * Weight)
        adjusted_val = v + (gap * weight)
        
        reconciled[m] = max(0.0, round(adjusted_val, 1))
        
    return reconciled

print('\nWLS reconciliation defined.')

Per-model forecast variances (lower = more reliable):
  Bolt                            σ²=      44.2
  Harrier                         σ²=      50.4
  Curvv                           σ²=      51.3
  Punch                           σ²=      54.8
  Safari                          σ²=      57.5
  Indigo                          σ²=      58.1
  Nexon EV                        σ²=      67.3
  Tiago                           σ²=      73.9
  Tiago EV                        σ²=      74.3
  Nexon                           σ²=      75.0
  Zest                            σ²=      84.2
  Tigor                           σ²=      92.3
  Nano                            σ²=     101.9
  Indica                          σ²=     111.9
  Altroz                          σ²=     115.8
  Safari Storme                   σ²=     145.1
  Hexa                            σ²=     202.2
  Tigor EV                        σ²=     251.5

WLS reconciliation defined.


## 13. Inference — Z-Scaled → Inverse Transform → Raw PO Units

**Step-by-step for each car model:**
1. Build features in Z-scaled space
2. LightGBM predicts a Z-score (e.g., `+0.5`)
3. `raw_units = z * σ_model + μ_model`
4. `raw_units = max(0, raw_units)` — no negative purchase orders
5. Apply bias correction × factor
6. WLS reconciliation to anchor to OEM total

In [20]:
def build_inference_row(car_model, model_monthly, model_enc, horizon, feat_names):
    """Build a single-row feature vector for the latest month."""
    
    sp = scaling_params[car_model]
    s = (
        model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
        .sort_values('ds').reset_index(drop=True)
    )
    full_r = pd.DataFrame({'ds': pd.date_range(s.ds.min(), s.ds.max(), freq='MS')})
    s = full_r.merge(s, on='ds', how='left').assign(y=lambda x: x['y'].fillna(0))
    
    if len(s) < 18:
        return None

    s['is_structural_zero'] = (s['y'] == 0).astype(int)
    s['y_scaled']           = s['y'] / sp['max']
    s['month']              = s['ds'].dt.month
    s['quarter']            = s['ds'].dt.quarter
    s['year']               = s['ds'].dt.year
    s['trend']              = np.arange(len(s))
    s['model_enc']          = model_enc[car_model]
    s['model_cluster']      = MODEL_CLUSTER[car_model]
    s['is_ev']              = get_ev_flag(car_model)
    s['bs6_era']            = bs6_flag(s['ds']).values
    s['fame2_active']       = fame2_flag(s['ds']).values
    s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
    s['is_q4_india']        = s['month'].isin([1, 2, 3]).astype(int)
    s['is_festive_season']  = s['month'].isin([9, 10, 11]).astype(int)
    
    for lag in [1, 2, 3, 4, 6, 9, 12, 24]:
        s[f'lag_{lag}'] = s['y_scaled'].shift(lag)
        
    for w in [3, 6, 12]:
        rolled = s['y_scaled'].shift(1).rolling(w)
        s[f'roll_mean_{w}'] = rolled.mean()
        s[f'roll_std_{w}']  = rolled.std()
        s[f'roll_max_{w}']  = rolled.max()
        
    s['ewm_slow']     = s['y_scaled'].shift(1).ewm(alpha=0.3, adjust=False).mean()
    s['ewm_fast']     = s['y_scaled'].shift(1).ewm(alpha=0.6, adjust=False).mean()
    s['ewm_diff']     = s['ewm_fast'] - s['ewm_slow']
    s['acceleration'] = s['lag_1'] - s['lag_2']
    s['yoy_scaled']   = s['y_scaled'].shift(12).pct_change(1)
    s['lag12_scaled'] = s['y_scaled'].shift(12)
    s['horizon']      = horizon

    # Fill missing feat columns with 0
    for f in feat_names:
        if f not in s.columns:
            s[f] = 0.0

    row = s[feat_names].iloc[[-1]]
    return row


def predict_reconciled_v6(model_monthly, global_models, oem_models, oem, bias_corrections, model_variances):
    """Generates 12-month PO forecasts using Variance-Weighted Reconciliation"""
    
    all_models = sorted(model_monthly['model'].unique())
    model_enc  = {m: i for i, m in enumerate(all_models)}
    records    = []
    last_date  = oem['ds'].max()

    for h in HORIZONS:
        if h not in global_models or h not in oem_models:
            continue
            
        gm   = global_models[h]
        feat = gm['feat_names']
        
        # Safe categorical index retrieval
        cidx = gm.get('cat_idx', 'auto')

        # OEM anchor (raw units — no scaling needed)
        X_oem, _, _, _ = build_oem_features(oem, h)
        oem_fc = float(np.clip(oem_models[h].predict(X_oem[[-1]])[0], 0, None))

        # Raw bottom-up (in Relative-space → inverse transform → raw units)
        bu_raw = {}
        for car_model in all_models:
            row = build_inference_row(car_model, model_monthly, model_enc, h, feat)
            
            if row is None:
                bu_raw[car_model] = 0.0
                continue
                
            sp = scaling_params[car_model]
            z_pred = float(gm['model'].predict(row.values, categorical_feature=cidx)[0])
            
            # ── CORRECTED MAX INVERSE TRANSFORM ──
            raw_pred = z_pred * sp['max']
            raw_pred = max(0.0, raw_pred)   # no negative POs
            bu_raw[car_model] = raw_pred

        # WLS reconciliation
        reconciled = reconcile_wls(bu_raw, oem_fc, model_variances, bias_corrections)
        forecast_date = (last_date + pd.DateOffset(months=h)).strftime('%Y-%m')

        for car_model in all_models:
            records.append({
                'model':            car_model,
                'forecast_month':   forecast_date,
                'horizon_h':        h,
                'raw_bu_units':     round(bu_raw.get(car_model, 0), 1),
                'oem_anchor':       round(oem_fc, 1),
                'reconciled_units': reconciled.get(car_model, 0.0),
            })

    return pd.DataFrame(records)


# Execute the Final Pipeline!
forecast_df = predict_reconciled_v6(
    model_monthly, global_models, oem_models, oem,
    bias_corrections, model_variances
)

print(forecast_df.sort_values(['forecast_month', 'model']).to_string(index=False))

        model forecast_month  horizon_h  raw_bu_units  oem_anchor  reconciled_units
       Altroz        2025-11          1          23.9       393.9              30.3
         Bolt        2025-11          1          17.3       393.9              13.6
        Curvv        2025-11          1          25.1       393.9              20.6
      Harrier        2025-11          1          21.2       393.9              18.3
         Hexa        2025-11          1          21.2       393.9              18.3
       Indica        2025-11          1          19.6       393.9              20.0
       Indigo        2025-11          1          19.6       393.9              23.2
         Nano        2025-11          1          20.6       393.9              22.9
        Nexon        2025-11          1          24.0       393.9              21.5
     Nexon EV        2025-11          1          17.3       393.9              19.3
        Punch        2025-11          1          25.4       393.9           

## 14. Evaluation — WAPE in Raw Unit Space (most honest metric)

In [22]:
def evaluate_v6(model_monthly, global_models, oem_models, oem,
                bias_corrections, model_variances, test_months=6):
    all_models = sorted(model_monthly['model'].unique())
    model_enc  = {m: i for i, m in enumerate(all_models)}
    rows = []
    
    for h in HORIZONS:
        if h not in global_models or h not in oem_models:
            continue
        gm   = global_models[h]
        feat = gm['feat_names']
        
        # Safely retrieve categorical index
        cidx = gm.get('cat_idx', 'auto')

        X_oem_all, y_oem_all, ds_oem, _ = build_oem_features(oem, h)
        oem_te_idx = np.arange(len(X_oem_all) - test_months, len(X_oem_all))
        oem_preds  = np.clip(oem_models[h].predict(X_oem_all[oem_te_idx]), 0, None)
        oem_actual = y_oem_all[oem_te_idx]

        bu_raw_matrix = {m: [] for m in all_models}
        actual_matrix = {m: [] for m in all_models}

        for car_model in all_models:
            sp = scaling_params[car_model]
            s = (
                model_monthly[model_monthly['model'] == car_model][['ds', 'y']]
                .sort_values('ds').reset_index(drop=True)
            )
            full_r = pd.DataFrame({'ds': pd.date_range(s.ds.min(), s.ds.max(), freq='MS')})
            s = full_r.merge(s, on='ds', how='left').assign(y=lambda x: x['y'].fillna(0))
            if len(s) < 18:
                continue

            s['is_structural_zero'] = (s['y'] == 0).astype(int)
            
            # ── MAX SCALING ──
            s['y_scaled'] = s['y'] / sp['max']
            
            s['month']   = s['ds'].dt.month; s['quarter'] = s['ds'].dt.quarter
            s['year']    = s['ds'].dt.year;  s['trend']   = np.arange(len(s))
            s['model_enc']      = model_enc[car_model]
            s['model_cluster']  = MODEL_CLUSTER[car_model]
            s['is_ev']          = get_ev_flag(car_model)
            s['bs6_era']        = bs6_flag(s['ds']).values
            s['fame2_active']   = fame2_flag(s['ds']).values
            s['festival_intensity'] = s['month'].map(FESTIVAL_MAP).fillna(0)
            s['is_q4_india']        = s['month'].isin([1, 2, 3]).astype(int)
            s['is_festive_season']  = s['month'].isin([9, 10, 11]).astype(int)
            
            for lag in [1, 2, 3, 4, 6, 9, 12, 24]:
                s[f'lag_{lag}'] = s['y_scaled'].shift(lag)
            for w in [3, 6, 12]:
                rolled = s['y_scaled'].shift(1).rolling(w)
                s[f'roll_mean_{w}'] = rolled.mean()
                s[f'roll_std_{w}']  = rolled.std()
                s[f'roll_max_{w}']  = rolled.max()
                
            s['ewm_slow']    = s['y_scaled'].shift(1).ewm(alpha=0.3, adjust=False).mean()
            s['ewm_fast']    = s['y_scaled'].shift(1).ewm(alpha=0.6, adjust=False).mean()
            s['ewm_diff']    = s['ewm_fast'] - s['ewm_slow']
            s['acceleration']  = s['lag_1'] - s['lag_2']
            s['yoy_scaled']    = s['y_scaled'].shift(12).pct_change(1)
            s['lag12_scaled']  = s['y_scaled'].shift(12)
            s['horizon'] = h
            s['target']  = s['y_scaled'].shift(-h)
            req = [f'lag_{l}' for l in [1, 2, 3, 6]]
            s = s.dropna(subset=['target'] + req).reset_index(drop=True)
            if len(s) < test_months + 1:
                continue

            te = s.iloc[-test_months:].copy()
            for mf in [f for f in feat if f not in te.columns]:
                te[mf] = 0.0

            z_pred = gm['model'].predict(te[feat].values, categorical_feature=cidx)
            
            # ── MAX INVERSE TRANSFORM ──
            raw_pred   = np.clip(z_pred * sp['max'], 0, None)
            raw_actual = np.clip(te['target'].values * sp['max'], 0, None)

            bu_raw_matrix[car_model] = list(raw_pred)
            actual_matrix[car_model] = list(raw_actual)

        active  = [m for m in all_models if len(actual_matrix.get(m, [])) == test_months]
        bu_arr  = np.array([bu_raw_matrix[m] for m in active])
        act_arr = np.array([actual_matrix[m]  for m in active])

        # WLS reconciled
        recon_arr = np.zeros_like(bu_arr)
        for t in range(test_months):
            bu_t     = {m: bu_arr[i, t] for i, m in enumerate(active)}
            recon_t  = reconcile_wls(bu_t, oem_preds[t], model_variances, bias_corrections)
            for i, m in enumerate(active):
                recon_arr[i, t] = recon_t[m]

        rows.append({
            'horizon':            h,
            'wape_raw_bu':        round(wape(act_arr.ravel(), bu_arr.ravel()),    2),
            'wape_reconciled':    round(wape(act_arr.ravel(), recon_arr.ravel()), 2),
            'wape_oem_anchor':    round(wape(oem_actual, oem_preds),              2),
        })

    return pd.DataFrame(rows)

# Evaluate and print results
eval_df = evaluate_v6(model_monthly, global_models, oem_models, oem,
                       bias_corrections, model_variances)

print(eval_df.to_string(index=False))
print(f"\nMean WAPE — Raw BU: {eval_df['wape_raw_bu'].mean():.1f}%  "
      f"Reconciled: {eval_df['wape_reconciled'].mean():.1f}%  "
      f"OEM anchor: {eval_df['wape_oem_anchor'].mean():.1f}%")

target_ok = eval_df['wape_reconciled'].mean() <= 15.0
fallback_ok = eval_df['wape_reconciled'].mean() <= 20.0
print(f"\n{'✅ TARGET MET (≤15%)' if target_ok else ('⚠ Fallback met (≤20%)' if fallback_ok else '❌ Above 20% — investigate top violators')}")

 horizon  wape_raw_bu  wape_reconciled  wape_oem_anchor
       1        34.88            34.21            19.04
       2        34.79            35.26            20.22
       3        36.36            34.09            11.63
       4        35.15            35.03            21.58
       5        33.36            36.34            27.63
       6        36.68            33.89            19.23
       7        36.67            36.00            19.80
       8        37.01            33.04            19.09
       9        36.55            34.13            12.39
      10        37.41            35.23            17.68
      11        37.25            34.04            17.39
      12        37.14            34.35            12.16

Mean WAPE — Raw BU: 36.1%  Reconciled: 34.6%  OEM anchor: 18.2%

❌ Above 20% — investigate top violators


## 15. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

# WAPE vs horizon
axes[0].plot(eval_df['horizon'], eval_df['wape_raw_bu'],      'o--', label='Raw BU (Z-scaled global)', alpha=0.7)
axes[0].plot(eval_df['horizon'], eval_df['wape_reconciled'],  's-',  label='WLS Reconciled', lw=2.5)
axes[0].plot(eval_df['horizon'], eval_df['wape_oem_anchor'],  'D:',  label='OEM anchor', lw=1.5)
axes[0].axhline(15, color='green', ls=':', lw=1.5, label='Target 15%')
axes[0].axhline(20, color='red',   ls=':', lw=1.5, label='Fallback 20%')
axes[0].set_xlabel('Horizon (months ahead)')
axes[0].set_ylabel('WAPE (%)')
axes[0].set_title('WAPE vs Horizon — v6 Z-scaled + WLS Reconciliation')
axes[0].set_xticks(HORIZONS)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Bias correction factors
models_sorted = sorted(bias_corrections.keys())
factors = [bias_corrections[m] for m in models_sorted]
colors  = ['green' if 0.85 <= f <= 1.15 else 'tomato' for f in factors]
axes[1].barh(models_sorted, factors, color=colors, alpha=0.8)
axes[1].axvline(1.0, color='black', lw=1)
axes[1].axvline(0.85, color='green', ls='--', lw=0.8)
axes[1].axvline(1.15, color='green', ls='--', lw=0.8)
axes[1].set_title('Per-Model Bias Correction Factors')
axes[1].set_xlabel('Multiplicative Factor (1.0 = unbiased)')

# 12-month reconciled vs historical
oem_rc = (
    forecast_df.groupby('forecast_month')['reconciled_units']
    .sum().reset_index().sort_values('forecast_month')
)
hist24 = oem.tail(24)
axes[2].plot(hist24['ds'], hist24['y'], 'o-', lw=1.5, ms=4, label='Historical')
axes[2].plot(pd.to_datetime(oem_rc['forecast_month']),
             oem_rc['reconciled_units'], 's--', lw=2, ms=6,
             color='tomato', label='WLS Reconciled Forecast')
axes[2].set_title('OEM Total — v6 WLS Reconciled Forecast')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Units')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
plt.xticks(rotation=20)

plt.tight_layout()
plt.savefig('wape_v6.png', dpi=120, bbox_inches='tight')
plt.show()


## 16. SHAP Feature Importance — Global Model h=1

In [ ]:
try:
    import shap
    gm1  = global_models[1]['model']
    X1, *_ = build_global_panel_zscaled(model_monthly, horizon=1)
    feat1 = global_models[1]['feat_names']
    cidx1 = global_models[1]['cat_idx']
    explainer   = shap.TreeExplainer(gm1)
    shap_values = explainer.shap_values(X1)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X1, feature_names=feat1, plot_type='bar', show=False)
    plt.title('SHAP — Global Regression LightGBM h=1 (Z-scaled features)')
    plt.tight_layout()
    plt.savefig('shap_v6_h1.png', dpi=120, bbox_inches='tight')
    plt.show()
except ImportError:
    print('pip install shap — skipping SHAP plot')


## 17. Save Outputs

In [ ]:
forecast_df.to_csv('per_model_forecast_v6.csv', index=False)

oem_rc_out = (
    forecast_df.groupby('forecast_month')
    .agg(
        raw_bu_total=('raw_bu_units', 'sum'),
        oem_anchor  =('oem_anchor',   'first'),
        reconciled_total=('reconciled_units', 'sum')
    )
    .reset_index().sort_values('forecast_month')
)
oem_rc_out.to_csv('oem_forecast_v6.csv', index=False)

import json
with open('scaling_params_v6.json', 'w') as f:
    json.dump(scaling_params, f, indent=2)
with open('bias_corrections_v6.json', 'w') as f:
    json.dump(bias_corrections, f, indent=2)

print('Saved:')
print('  per_model_forecast_v6.csv')
print('  oem_forecast_v6.csv')
print('  scaling_params_v6.json')
print('  bias_corrections_v6.json')
print(oem_rc_out.to_string(index=False))


## 18. Architecture Summary & Further Steps

| Component | v5 | v6 |
|---|---|---|
| Target variable | Raw units | **Z-score per model** |
| Objective | Tweedie | **Regression (L2)** |
| Lag features | On raw `y` | **On Z-scaled `y`** |
| Sample weights | None | **Exponential recency decay** |
| Reconciliation | Proportional | **WLS (inverse-variance)** |
| Bias correction | None | **Per-model multiplicative** |
| Clustering | None | **KMeans(k=4) cluster feature** |
| EWM features | None | **α=0.3 slow, α=0.6 fast, crossover** |
| Structural zeros | Filled with 0 | **Flagged + filled** |
| Optuna trials | 60 | **80 + max_depth tuning** |
